<a href="https://colab.research.google.com/github/Hendy1604/UAS_PraktikumNLP/blob/main/Uas__PrakNLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏋️ Gym Assistant — RAG Chatbot dengan LangChain, LangGraph & LangSmith

Notebook ini membangun chatbot **personal trainer gym** berbasis **RAG (Retrieval-Augmented Generation)**:
pertanyaan pengguna dicari kecocokannya di dataset latihan gym (`megaGymDataset.csv`), lalu hasil pencarian
dipakai sebagai konteks untuk LLM (Groq — Llama 3.3 70B) dalam menyusun jawaban.

## Apa Bedanya LangChain, LangGraph, dan LangSmith?

| Library | Fungsi Utama | Peran di Notebook Ini |
|---|---|---|
| **LangChain** | Framework "lego" untuk membangun aplikasi berbasis LLM. Menyediakan komponen siap pakai: wrapper LLM (`ChatGroq`), wrapper embedding, vector store (`FAISS`), `Document`, dan `ChatPromptTemplate`. Tugasnya menyatukan potongan-potongan ini jadi satu alur. | Menyediakan semua "bahan baku": memuat dokumen, membuat embedding, menyimpan/mencari vektor (retriever), dan memanggil LLM. |
| **LangGraph** | Lapisan **orkestrasi** di atas LangChain untuk alur kerja yang **stateful** dan **bercabang**, digambarkan sebagai graf (node + edge), bukan sekadar rantai linear. Cocok untuk alur multi-langkah, loop, atau percabangan kondisional. | Mendefinisikan alur RAG sebagai graf 2 node: `retrieve` → `generate`. State (`GymState`) mengalir dan diperkaya di setiap node. |
| **LangSmith** | Platform **observability & debugging** untuk aplikasi LLM. Mencatat (trace) setiap pemanggilan LLM/chain: input, output, durasi, token, dan error — bisa dilihat di dashboard web. | Setelah `LANGCHAIN_TRACING_V2=true` diaktifkan, setiap `graph.invoke()` otomatis terkirim ke dashboard LangSmith untuk dipantau dan dievaluasi. |



## 1. Instalasi Dependensi



In [ ]:
!pip install -q -U langchain langgraph langsmith langchain-community langchain-groq langchain-huggingface
!pip install -q faiss-cpu sentence-transformers gradio


## 2. Import & Konfigurasi Dasar

In [ ]:
import os
import time
import getpass
from typing import TypedDict, List

import pandas as pd
import gradio as gr

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END


/tmp/ipykernel_17168/494853667.py:12: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 3. Memuat Dataset



In [ ]:
CSV_PATH = "megaGymDataset.csv"
REQUIRED_COLUMNS = ["Title", "Desc", "BodyPart", "Equipment", "Level", "Type"]

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(
        f"File '{CSV_PATH}' tidak ditemukan. Unggah file dataset terlebih dahulu "
        f"(Colab: panel folder di kiri > klik kanan > Upload)."
    )

df = pd.read_csv(CSV_PATH)

missing_cols = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing_cols:
    raise ValueError(f"Kolom berikut tidak ditemukan di CSV: {missing_cols}")

df = df.dropna(subset=["Title"]).fillna("Tidak diketahui")

documents: List[Document] = []
for _, row in df.iterrows():
    content = f"""Exercise : {row['Title']}
Description : {row['Desc']}
Body Part : {row['BodyPart']}
Equipment : {row['Equipment']}
Level : {row['Level']}
Type : {row['Type']}"""
    documents.append(
        Document(
            page_content=content,
            # Metadata terstruktur ini dipakai nanti untuk filter cepat
            # (bukan cuma similarity search), supaya hasil retrieval lebih akurat.
            metadata={
                "title": str(row["Title"]),
                "body_part": str(row["BodyPart"]),
                "equipment": str(row["Equipment"]),
                "level": str(row["Level"]),
            },
        )
    )

print(f"Jumlah dokumen latihan yang dimuat: {len(documents)}")
print("Contoh nilai unik Body Part:", sorted(df["BodyPart"].unique())[:10], "...")
print("Contoh nilai unik Equipment:", sorted(df["Equipment"].unique()))
print("Contoh nilai unik Level:", sorted(df["Level"].unique()))


Jumlah dokumen latihan yang dimuat: 2918
Contoh nilai unik Body Part: ['Abdominals', 'Abductors', 'Adductors', 'Biceps', 'Calves', 'Chest', 'Forearms', 'Glutes', 'Hamstrings', 'Lats'] ...
Contoh nilai unik Equipment: ['Bands', 'Barbell', 'Body Only', 'Cable', 'Dumbbell', 'E-Z Curl Bar', 'Exercise Ball', 'Foam Roll', 'Kettlebells', 'Machine', 'Medicine Ball', 'Other', 'Tidak diketahui']
Contoh nilai unik Level: ['Beginner', 'Expert', 'Intermediate']


## 4. Embedding & Vector Store (FAISS)

Mengubah setiap dokumen latihan menjadi vektor numerik (embedding), lalu menyimpannya di FAISS agar bisa
dicari berdasarkan kemiripan makna (semantic search). `retriever` hanya dibuat **sekali** di sini
.

 **Catatan penting:** dataset ini berisi teks **bahasa Inggris** (`Chest`, `Quadriceps`, dst), sedangkan
pertanyaan pengguna kemungkinan besar **bahasa Indonesia** ("latihan dada"). Model embedding `all-MiniLM-L6-v2`
versi asli dilatih dominan untuk Inggris, sehingga pencarian "dada" sering tidak match dengan "Chest" dan
malah mengambil body part lain yang kebetulan mirip secara vektor. Di sini kita pakai model **multilingual**
(`paraphrase-multilingual-MiniLM-L12-v2`) agar pencarian lintas bahasa Indonesia↔Inggris lebih akurat.

In [ ]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

vectorstore = FAISS.from_documents(documents, embedding)

retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

print("Vector store siap. Contoh pencarian:")
for doc in retriever.invoke("latihan dada untuk pemula"):
    print("-", doc.metadata["title"], "|", doc.metadata["body_part"])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector store siap. Contoh pencarian:
- Chest Push with Run Release | Chest
- Chest Push (single response) | Chest
- Heavy Bag Thrust | Chest
- Forward Drag with Press | Chest
- Leg-Over Floor Press | Chest
- 30 Chest 30-Degree Incline Dumbbell Press | Chest
- Chest Push (multiple response) | Chest
- Holman Push-Up to Belly Touch | Chest


## 5. Mengaktifkan LangSmith (Monitoring & Tracing)



In [ ]:
from langchain_core.tracers.langchain import wait_for_all_tracers

ENABLE_LANGSMITH = True  # ubah ke False jika tidak ingin tracing
LANGSMITH_ENDPOINT = "https://api.smith.langchain.com"  # ganti ke https://eu.api.smith.langchain.com jika akun Anda di region EU
LANGSMITH_PROJECT_NAME = "GymAssistant"

if ENABLE_LANGSMITH:
    if "LANGCHAIN_API_KEY" not in os.environ or not os.environ["LANGCHAIN_API_KEY"]:
        # .strip() untuk membuang spasi/newline tak sengaja yang ikut ter-paste
        os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("Masukkan LangSmith API Key: ").strip()

    # Set kedua skema nama env var (lama "LANGCHAIN_*" & baru "LANGSMITH_*") supaya kompatibel
    # dengan versi SDK yang berbeda-beda.
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_ENDPOINT"] = LANGSMITH_ENDPOINT
    os.environ["LANGCHAIN_PROJECT"] = LANGSMITH_PROJECT_NAME

    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = os.environ["LANGCHAIN_API_KEY"]
    os.environ["LANGSMITH_ENDPOINT"] = LANGSMITH_ENDPOINT
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT_NAME

    print(f"🔧 Tracing diaktifkan — project: '{LANGSMITH_PROJECT_NAME}', endpoint: {LANGSMITH_ENDPOINT}")
    print("   Lanjutkan ke sel 'Uji Coba Cepat' di bawah, lalu cek dashboard: https://smith.langchain.com/")
else:
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing NONAKTIF")


🔧 Tracing diaktifkan — project: 'GymAssistant', endpoint: https://api.smith.langchain.com
   Lanjutkan ke sel 'Uji Coba Cepat' di bawah, lalu cek dashboard: https://smith.langchain.com/


## 6. Konfigurasi LLM (Groq — Gratis)




In [ ]:
if "GROQ_API_KEY" not in os.environ or not os.environ["GROQ_API_KEY"]:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Masukkan Groq API Key: ")

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3,
)

print("LLM aktif:", llm.model_name)


LLM aktif: llama-3.3-70b-versatile


## 7. Prompt Template

Template ini didefinisikan **sekali** dan benar-benar dipakai di node `generate` (di versi asli, template
ini dibuat tapi tidak pernah digunakan — `generate()` malah membangun prompt f-string terpisah).

In [ ]:
prompt = ChatPromptTemplate.from_template(
"""Kamu adalah personal trainer gym profesional dan ramah.

Gunakan informasi latihan berikut sebagai konteks (jangan mengarang latihan di luar konteks ini):

{context}

Pertanyaan pengguna:
{question}

Berikan jawaban terstruktur:
1. Nama latihan
2. Deskripsi singkat
3. Body part yang dilatih
4. Equipment yang dibutuhkan
5. Tingkat kesulitan

Jika tidak ada latihan yang relevan di konteks, katakan dengan jujur bahwa kamu tidak menemukannya."""
)

rag_chain = prompt | llm


## 8. State Graph (LangGraph)

`GymState` mendefinisikan data yang mengalir antar node dalam graf.

In [ ]:
class GymState(TypedDict):
    question: str
    context: list
    answer: str
    retrieved_docs: str
    final_prompt: str


## Kamus Kata Kunci (Indonesia → Inggris)

Untuk meningkatkan akurasi, saya tambahkan **retrieval hibrida**: selain pencarian vektor (semantic search),
kita juga mencocokkan kata kunci umum Bahasa Indonesia langsung ke nilai kolom `BodyPart`, `Equipment`,
dan `Level` di dataset. Jika query mengandung kata kunci ini, dokumen yang cocok metadata-nya akan
diprioritaskan sebelum digabung dengan hasil pencarian vektor.

In [ ]:
BODY_PART_KEYWORDS = {
    "dada": ["Chest"],
    "chest": ["Chest"],
    "punggung": ["Lats", "Middle Back", "Lower Back"],
    "back": ["Lats", "Middle Back", "Lower Back"],
    "kaki": ["Quadriceps", "Hamstrings", "Calves", "Glutes", "Adductors", "Abductors"],
    "paha": ["Quadriceps", "Hamstrings", "Adductors", "Abductors"],
    "betis": ["Calves"],
    "perut": ["Abdominals"],
    "abs": ["Abdominals"],
    "bahu": ["Shoulders"],
    "pundak": ["Shoulders"],
    "lengan": ["Biceps", "Triceps", "Forearms"],
    "bisep": ["Biceps"],
    "trisep": ["Triceps"],
    "tangan": ["Forearms", "Biceps", "Triceps"],
    "bokong": ["Glutes"],
    "glutes": ["Glutes"],
    "leher": ["Neck"],
    "trapezius": ["Traps"],
    "trapesius": ["Traps"],
}

EQUIPMENT_KEYWORDS = {
    "tanpa alat": ["Body Only", "None"],
    "tanpa peralatan": ["Body Only", "None"],
    "bodyweight": ["Body Only", "None"],
    "di rumah": ["Body Only", "None"],
}

LEVEL_KEYWORDS = {
    "pemula": ["Beginner"],
    "dasar": ["Beginner"],
    "menengah": ["Intermediate"],
    "lanjutan": ["Expert"],
    "mahir": ["Expert"],
    "expert": ["Expert"],
}


def _match_keywords(question: str, keyword_map: dict) -> set:
    q = question.lower()
    matched = set()
    for kw, values in keyword_map.items():
        if kw in q:
            matched.update(values)
    return matched


def hybrid_retrieve(question: str, k: int = 8) -> list:
    """Gabungkan filter metadata (kata kunci ID) dengan pencarian vektor."""
    body_parts = _match_keywords(question, BODY_PART_KEYWORDS)
    equipments = _match_keywords(question, EQUIPMENT_KEYWORDS)
    levels = _match_keywords(question, LEVEL_KEYWORDS)

    filtered_docs = []
    if body_parts or equipments or levels:
        for doc in documents:
            md_ = doc.metadata
            if body_parts and md_["body_part"] not in body_parts:
                continue
            if equipments and md_["equipment"] not in equipments:
                continue
            if levels and md_["level"] not in levels:
                continue
            filtered_docs.append(doc)

    # Pencarian vektor sebagai pelengkap/cadangan
    vector_docs = retriever.invoke(question)

    # Gabungkan: hasil filter metadata diprioritaskan, lalu lengkapi dengan hasil vektor (dedup by title)
    combined, seen_titles = [], set()
    for doc in (filtered_docs + vector_docs):
        title = doc.metadata.get("title")
        if title in seen_titles:
            continue
        seen_titles.add(title)
        combined.append(doc)
        if len(combined) >= k:
            break

    return combined


def retrieve(state: GymState) -> dict:
    """Node 1: ambil dokumen latihan yang relevan (hybrid: filter metadata + vektor)."""
    docs = hybrid_retrieve(state["question"], k=8)
    retrieved_docs = "\n\n".join(doc.page_content for doc in docs)
    return {
        "context": docs,
        "retrieved_docs": retrieved_docs,
    }


def generate(state: GymState) -> dict:
    """Node 2: susun jawaban dengan LLM, memakai prompt template + konteks hasil retrieve."""
    context = "\n\n".join(doc.page_content for doc in state["context"])

    try:
        response = rag_chain.invoke({
            "context": context,
            "question": state["question"],
        })
        answer = response.content
    except Exception as e:
        answer = f"⚠️ Terjadi error saat memanggil LLM: {e}"

    final_prompt = prompt.format(context=context, question=state["question"])

    return {
        "answer": answer,
        "final_prompt": final_prompt,
    }


## 9. Merangkai Graph

In [ ]:
builder = StateGraph(GymState)

builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)

graph = builder.compile()


## 10. Uji Coba Cepat

Setelah `invoke()`, kita panggil `wait_for_all_tracers()` agar trace benar-benar terkirim ke LangSmith
sebelum sel selesai dieksekusi — tanpa ini, trace mungkin belum sampai saat Anda membuka dashboard.

In [ ]:
result = graph.invoke({"question": "latihan dada untuk pemula"})
wait_for_all_tracers()
print(result["answer"])


Berikut beberapa latihan dada untuk pemula yang saya temukan:

1. **Cross Over - With Bands**
 * Deskripsi singkat: Tidak diketahui
 * Body part yang dilatih: Dada (Chest)
 * Equipment yang dibutuhkan: Bands
 * Tingkat kesulitan: Pemula (Beginner)

2. **Bench Press - With Bands**
 * Deskripsi singkat: Tidak diketahui
 * Body part yang dilatih: Dada (Chest)
 * Equipment yang dibutuhkan: Bands
 * Tingkat kesulitan: Pemula (Beginner)

3. **Bench Press With Short Bands**
 * Deskripsi singkat: Tidak diketahui
 * Body part yang dilatih: Dada (Chest)
 * Equipment yang dibutuhkan: Bands
 * Tingkat kesulitan: Pemula (Beginner)

4. **Feet-Elevated TRX Push-Up**
 * Deskripsi singkat: Tidak diketahui
 * Body part yang dilatih: Dada (Chest)
 * Equipment yang dibutuhkan: Bands
 * Tingkat kesulitan: Pemula (Beginner)

5. **Wide-grip bench press**
 * Deskripsi singkat: Latihan ini menargetkan dada dan trisep, dengan meletakkan tangan lebih jauh di barbell.
 * Body part yang dilatih: Dada (Chest)
 * Eq

## 11. Fungsi Chat (dengan Monitoring Waktu Eksekusi)

In [ ]:
def chat_monitor(message: str, history: list):
    """Memanggil graph RAG, lalu mengembalikan history chat + panel tracing."""
    if not message or not message.strip():
        return history, "_Masukkan pertanyaan terlebih dahulu._"

    history = history or []
    start = time.time()

    try:
        result = graph.invoke({"question": message})
        wait_for_all_tracers()  # pastikan trace terkirim ke LangSmith sebelum lanjut
        answer = result["answer"]
        retrieved = result.get("retrieved_docs", "Tidak tersedia")
        final_prompt = result.get("final_prompt", "Tidak tersedia")
        status = "✅ Berhasil"
    except Exception as e:
        answer = f"⚠️ Maaf, terjadi kesalahan: {e}"
        retrieved = "Tidak tersedia (terjadi error)"
        final_prompt = "Tidak tersedia (terjadi error)"
        status = "❌ Error"

    elapsed = round(time.time() - start, 2)
    ls_link = "https://smith.langchain.com/" if os.environ.get("LANGCHAIN_TRACING_V2") == "true" else None

    trace = f"""### 📊 Monitoring & Tracing

**Status:** {status}  **Waktu eksekusi:** {elapsed} detik
{f"**LangSmith:** [Buka dashboard]({ls_link})" if ls_link else "**LangSmith:** tracing nonaktif"}

---
**Pertanyaan pengguna**

{message}

---
**Dokumen yang diambil (retrieved)**

{retrieved}

---
**Prompt final ke LLM (Groq)**

```
{final_prompt}
```
"""

    history.append({"role": "user", "content": message})
    history.append({"role": "assistant", "content": answer})

    return history, trace


def clear_chat():
    return [], "Tracing akan muncul di sini setelah Anda mengirim pertanyaan."


## 12. UI Gradio — Versi yang Sudah Di-upgrade



In [ ]:
from langsmith import Client
import uuid
from datetime import datetime, timezone
import time

client = Client()
run_id = str(uuid.uuid4())
project = os.environ.get("LANGCHAIN_PROJECT", "default")
print("Mengirim run_id:", run_id, "ke project:", project)

try:
    client.create_run(
        name="DIRECT-TEST",
        run_type="chain",
        inputs={"test": "halo"},
        id=run_id,
        project_name=project,
        start_time=datetime.now(timezone.utc),
    )
    print("✅ create_run() tidak melempar error")
except Exception as e:
    print("❌ create_run() GAGAL dengan error:", repr(e))

try:
    client.update_run(run_id, outputs={"result": "selesai"}, end_time=datetime.now(timezone.utc))
    print("✅ update_run() tidak melempar error")
except Exception as e:
    print("❌ update_run() GAGAL dengan error:", repr(e))

time.sleep(3)

try:
    r = client.read_run(run_id)
    print("✅✅ BERHASIL terverifikasi ada di server LangSmith:", r.id)
except Exception as e:
    print("❌ Tidak ditemukan di server saat dibaca balik:", repr(e))

Mengirim run_id: e7dfe863-6532-45ef-8f1f-a371fe813bf4 ke project: GymAssistant
2026-06-20 19:39:33,034 urllib3.connectionpool DEBUG: Starting new HTTPS connection (1): api.smith.langchain.com:443
2026-06-20 19:39:33,077 urllib3.connectionpool DEBUG: Starting new HTTPS connection (2): api.smith.langchain.com:443
2026-06-20 19:39:33,237 urllib3.connectionpool DEBUG: https://api.smith.langchain.com:443 "GET /info HTTP/1.1" 200 1239
2026-06-20 19:39:33,245 langsmith.client DEBUG: Tracing control thread func compress parallel called
2026-06-20 19:39:33,358 urllib3.connectionpool DEBUG: https://api.smith.langchain.com:443 "POST /runs HTTP/1.1" 202 26
✅ create_run() tidak melempar error
2026-06-20 19:39:33,522 urllib3.connectionpool DEBUG: https://api.smith.langchain.com:443 "PATCH /runs/e7dfe863-6532-45ef-8f1f-a371fe813bf4 HTTP/1.1" 202 26
✅ update_run() tidak melempar error
2026-06-20 19:39:38,949 urllib3.connectionpool DEBUG: https://api.smith.langchain.com:443 "GET /runs/e7dfe863-6532-45e

In [ ]:
custom_theme = gr.themes.Soft(
    primary_hue="emerald",
    secondary_hue="teal",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
).set(
    button_primary_background_fill="*primary_600",
    button_primary_background_fill_hover="*primary_700",
    block_radius="16px",
)

custom_css = """
#header-card {
    background: linear-gradient(135deg, #047857 0%, #0f766e 100%);
    color: white;
    border-radius: 16px;
    padding: 20px 24px;
    margin-bottom: 8px;
}
#header-card h1 { margin: 0 0 4px 0; }
#header-card p { margin: 0; opacity: 0.9; }
.gr-chatbot { border-radius: 16px !important; }
"""

EXAMPLE_QUESTIONS = [
    "Latihan dada untuk pemula",
    "Rekomendasi latihan kaki tanpa alat",
    "Latihan punggung untuk level menengah",
    "Latihan perut yang bisa dilakukan di rumah",
]

with gr.Blocks(theme=custom_theme, css=custom_css, title="Gym Assistant") as demo:

    gr.HTML(
        """
        <div id="header-card">
            <h1>🏋️ Gym Assistant</h1>
            <p>Tanyakan rekomendasi latihan gym — dijawab dengan RAG (LangChain + LangGraph) dari dataset latihan,
            dipantau lewat LangSmith.</p>
        </div>
        """
    )

    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Chat",
                type="messages",
                height=460,
                avatar_images=(None, "🏋️"),
                show_copy_button=True,
            )

            with gr.Row():
                msg = gr.Textbox(
                    label=None,
                    placeholder="Contoh: latihan dada untuk pemula...",
                    scale=5,
                    container=False,
                )
                send = gr.Button("Kirim", variant="primary", scale=1)

            with gr.Row():
                clear_btn = gr.Button("🗑️ Hapus Percakapan", size="sm")

            gr.Examples(
                examples=EXAMPLE_QUESTIONS,
                inputs=msg,
                label="Contoh pertanyaan",
            )

        with gr.Column(scale=2):
            with gr.Accordion("📊 Monitoring & Tracing (LangSmith)", open=False):
                trace_box = gr.Markdown(
                    value="Tracing akan muncul di sini setelah Anda mengirim pertanyaan."
                )

    # Submit lewat tombol atau menekan Enter
    send.click(fn=chat_monitor, inputs=[msg, chatbot], outputs=[chatbot, trace_box]).then(
        lambda: "", outputs=msg
    )
    msg.submit(fn=chat_monitor, inputs=[msg, chatbot], outputs=[chatbot, trace_box]).then(
        lambda: "", outputs=msg
    )
    clear_btn.click(fn=clear_chat, outputs=[chatbot, trace_box])

demo.launch(share=True, debug=True)


2026-06-20 19:36:54,716 httpcore.connection DEBUG: close.started
2026-06-20 19:36:54,737 httpcore.connection DEBUG: close.complete
2026-06-20 19:36:54,751 httpcore.connection DEBUG: connect_tcp.started host='huggingface.co' port=443 local_address=None timeout=None socket_options=None
2026-06-20 19:36:54,794 httpcore.connection DEBUG: connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7bb5d66fe120>
2026-06-20 19:36:54,796 httpcore.connection DEBUG: connect_tcp.started host='api.gradio.app' port=443 local_address=None timeout=3 socket_options=None
2026-06-20 19:36:54,801 httpcore.connection DEBUG: start_tls.started ssl_context=<ssl.SSLContext object at 0x7bb5e6d00c50> server_hostname='huggingface.co' timeout=None


/tmp/ipykernel_17168/717749249.py:32: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, css=custom_css, title="Gym Assistant") as demo:
/tmp/ipykernel_17168/717749249.py:32: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=custom_theme, css=custom_css, title="Gym Assistant") as demo:


2026-06-20 19:36:54,902 httpcore.connection DEBUG: start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7bb5d7b0a930>
2026-06-20 19:36:54,907 httpcore.http11 DEBUG: send_request_headers.started request=<Request [b'HEAD']>
2026-06-20 19:36:54,911 httpcore.http11 DEBUG: send_request_headers.complete
2026-06-20 19:36:54,913 httpcore.http11 DEBUG: send_request_body.started request=<Request [b'HEAD']>
2026-06-20 19:36:54,919 httpcore.http11 DEBUG: send_request_body.complete
2026-06-20 19:36:54,923 httpcore.http11 DEBUG: receive_response_headers.started request=<Request [b'HEAD']>
2026-06-20 19:36:55,011 httpcore.http11 DEBUG: receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'2'), (b'Connection', b'keep-alive'), (b'Date', b'Sat, 20 Jun 2026 19:36:54 GMT'), (b'ETag', b'W/"2-nOO9QiTIwXgNtWtBJezz8kv3SLc"'), (b'X-Powered-By', b'huggingface-moon'), (b'X-Request-Id', b'Root=1-6

/tmp/ipykernel_17168/717749249.py:46: DeprecationWarning: The 'show_copy_button' parameter will be removed in Gradio 6.0. You will need to use 'buttons=["copy"]' instead.
  chatbot = gr.Chatbot(
/tmp/ipykernel_17168/717749249.py:46: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


2026-06-20 19:36:55,096 httpcore.connection DEBUG: start_tls.started ssl_context=<ssl.SSLContext object at 0x7bb5d6648650> server_hostname='api.gradio.app' timeout=3
2026-06-20 19:36:55,183 httpcore.http11 DEBUG: receive_response_body.started request=<Request [b'HEAD']>
2026-06-20 19:36:55,233 httpcore.http11 DEBUG: receive_response_body.complete
2026-06-20 19:36:55,242 httpcore.http11 DEBUG: response_closed.started
2026-06-20 19:36:55,244 httpcore.http11 DEBUG: response_closed.complete
2026-06-20 19:36:55,306 httpcore.connection DEBUG: connect_tcp.started host='127.0.0.1' port=7860 local_address=None timeout=None socket_options=None
2026-06-20 19:36:55,309 httpcore.connection DEBUG: connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x7bb5d78cb230>
2026-06-20 19:36:55,311 httpcore.http11 DEBUG: send_request_headers.started request=<Request [b'GET']>
2026-06-20 19:36:55,315 httpcore.http11 DEBUG: send_request_headers.complete
2026-06-20 19:36:55,316 httpcor

2026-06-20 19:36:57,410 httpcore.http11 DEBUG: send_request_headers.started request=<Request [b'HEAD']>
2026-06-20 19:36:57,412 httpcore.http11 DEBUG: send_request_headers.complete
2026-06-20 19:36:57,413 httpcore.http11 DEBUG: send_request_body.started request=<Request [b'HEAD']>
2026-06-20 19:36:57,414 httpcore.http11 DEBUG: send_request_body.complete
2026-06-20 19:36:57,415 httpcore.http11 DEBUG: receive_response_headers.started request=<Request [b'HEAD']>
2026-06-20 19:36:57,511 httpcore.http11 DEBUG: receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Content-Type', b'text/plain; charset=utf-8'), (b'Content-Length', b'2'), (b'Connection', b'keep-alive'), (b'Date', b'Sat, 20 Jun 2026 19:36:57 GMT'), (b'ETag', b'W/"2-nOO9QiTIwXgNtWtBJezz8kv3SLc"'), (b'X-Powered-By', b'huggingface-moon'), (b'X-Request-Id', b'Root=1-6a36ebd9-33e61fc0271a2cdc01961e9f;50c32eff-0a4b-4f56-93d4-3a5d4aac8c0f'), (b'RateLimit', b'"api";r=9998;t=127'), (b'RateLimit-Policy', b'"fixed wi

In [ ]:
import logging, sys, os

# Paksa reset handler logging — basicConfig() biasa sering tidak berpengaruh di Colab
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)

handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(asctime)s %(name)s %(levelname)s: %(message)s"))
logging.root.addHandler(handler)
logging.root.setLevel(logging.DEBUG)

logging.getLogger("langsmith").setLevel(logging.DEBUG)
logging.getLogger("urllib3").setLevel(logging.DEBUG)  # supaya request HTTP mentah ke server kelihatan

# Cetak ulang env var (key disensor sebagian) untuk pastikan tidak stale setelah restart
key = os.environ.get("LANGCHAIN_API_KEY", "")
print("TRACING_V2 :", os.environ.get("LANGCHAIN_TRACING_V2"))
print("PROJECT    :", os.environ.get("LANGCHAIN_PROJECT"))
print("API KEY    :", key[:8] + "..." + key[-4:] if len(key) > 12 else "(kosong/pendek)")

import uuid
from langchain_core.tracers.langchain import wait_for_all_tracers
from langchain_core.runnables import RunnableLambda

test_id = str(uuid.uuid4())[:8]
run_name = f"DIAGNOSTIC2-{test_id}"
print(">>> mulai invoke...")
out = RunnableLambda(lambda x: f"test {test_id}").invoke("ping", config={"run_name": run_name})
print(">>> invoke selesai, menunggu tracer flush...")
wait_for_all_tracers()
print(">>> selesai. Cari run:", run_name)

TRACING_V2 : true
PROJECT    : GymAssistant
API KEY    : lsv2_pt_...8e4b
>>> mulai invoke...
>>> invoke selesai, menunggu tracer flush...
>>> selesai. Cari run: DIAGNOSTIC2-ebc4fd01


In [ ]:
import uuid
from langchain_core.tracers.langchain import wait_for_all_tracers
from langchain_core.runnables import RunnableLambda

print("LANGCHAIN_TRACING_V2 :", os.environ.get("LANGCHAIN_TRACING_V2"))
print("LANGCHAIN_PROJECT    :", os.environ.get("LANGCHAIN_PROJECT"))
print("LANGCHAIN_API_KEY set:", bool(os.environ.get("LANGCHAIN_API_KEY")))

test_id = str(uuid.uuid4())[:8]
run_name = f"DIAGNOSTIC-{test_id}"

test_chain = RunnableLambda(lambda x: f"halo dari test {test_id}")
out = test_chain.invoke("ping", config={"run_name": run_name})
wait_for_all_tracers()

print("Output:", out)
print("Cari run dengan nama ini di dashboard ->", run_name)

LANGCHAIN_TRACING_V2 : true
LANGCHAIN_PROJECT    : gymAssistant
LANGCHAIN_API_KEY set: True
Output: halo dari test f92e17aa
Cari run dengan nama ini di dashboard -> DIAGNOSTIC-f92e17aa


## Catatan Keamanan

- **Jangan** commit notebook ini ke repository publik setelah menjalankan sel `getpass` jika output sel
  tersebut sempat menampilkan key (seharusnya tidak, karena `getpass` menyembunyikan input).
- Jika API key pernah ter-paste langsung di sel kode/markdown sebelumnya (seperti pada versi asli notebook ini),
  **cabut (revoke) key tersebut** di Google AI Studio / LangSmith dan buat key baru, karena key lama
  sudah tersimpan di riwayat notebook/Git.